In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

**<font size="6" color="red">ch02. Ollama_LLM활용의 기본개념(LangChain)</form>**

# 1. LLM 을 활용하여 답변 생성


## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT(open ai API), Claude같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용
### ollama.com 설치 -> 모델 pull
- cmd창에서 ollama pull deepseek-r1:1.5b

In [7]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of France?')
result

AIMessage(content='\n\nThe capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-09-10T07:56:10.7838074Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14134282300, 'load_duration': 1932100, 'prompt_eval_count': 10, 'prompt_eval_duration': 34215000, 'eval_count': 458, 'eval_duration': 14091408000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a08a50-f567-7e52-8e54-56261e0aae48-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 458, 'total_tokens': 468})

In [8]:
print(result.content)



The capital of France is Paris.


### 모델 pull
- ollama run Llama-3.2-1B
- ollama 모델은 공식적으로 한글 지원을 하지 안 됨(llama

In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
result = llm.invoke('What is the capital of Korea?')
print(result.content)

The capital of Korea is Seoul.


In [2]:
result = llm.invoke('한국 수도는 어디에요?')
print(result.content)

한국의 수도는 Seoul이며, 수도가 아니라 수도권의 수도입니다. 수도권은 Seoul과 surrounding 지역을 포함하는 지역입니다.


## 2) open AI 모델 활용
- pip install langchain-openai

In [5]:
# 환경변수 ( `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.)
from dotenv import load_dotenv
import os
load_dotenv()
# print(os.getenv('OPENAI_API_KEY'))
# print(os.environ['OPENAI_API_KEY'])

True

In [6]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")
result = llm.invoke('What is the capital of Korea?')
# result = llm.invoke('한국의 수도가 어디예요?')
result.content

'The capital of South Korea is Seoul.'

In [7]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
# llm.invoke('What is the capital of Korea?')

TypeError: Anthropic authentication failed: no API key or authorization credentials were provided. Set the ANTHROPIC_API_KEY environment variable, pass api_key=... to ChatAnthropic, or provide credentials via default_headers={"Authorization": ...}. If you are routing through the LangSmith gateway, set LANGSMITH_GATEWAY and LANGSMITH_GATEWAY_API_KEY.

# 2. 렝체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질문

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate 을 사용하여 변수가 포함된 템플릿 작성하면 PromptValue를 만들 수 있다


In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0)
llm.invoke("What is the capital of Korea")
# 프롬프트 가능 타입 : str, PromptValue, list of BaseMessages

AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T02:23:48.379257Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2233534400, 'load_duration': 1628034800, 'prompt_eval_count': 31, 'prompt_eval_duration': 333207000, 'eval_count': 8, 'eval_duration': 267612000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e47-33de-77d2-8dff-94ae379605f5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 8, 'total_tokens': 39})

In [5]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
                        template="What is the capital of {country}?", # {}안에 값을 새로운 값으로 대체
                        input_variables = ['country']
                )
prompt = prompt_template.invoke({"country":"Korea"})
print(1, prompt)
prompt = prompt_template.invoke("Korea")
print(2, prompt)
llm.invoke(prompt)

1 text='What is the capital of Korea?'
2 text='What is the capital of Korea?'


AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T02:26:00.0171034Z', 'done': True, 'done_reason': 'stop', 'total_duration': 369318500, 'load_duration': 2073400, 'prompt_eval_count': 32, 'prompt_eval_duration': 116130000, 'eval_count': 8, 'eval_duration': 247663000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e49-3d5d-7670-9d1b-ea22cf123fd5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [8]:
country = input('수도를 알고 싶은 나라는(영어)? ')
llm.invoke(prompt_template.invoke(country))

수도를 알고 싶은 나라는(영어)? france


AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T02:36:23.8205799Z', 'done': True, 'done_reason': 'stop', 'total_duration': 361288700, 'load_duration': 3288700, 'prompt_eval_count': 32, 'prompt_eval_duration': 103750000, 'eval_count': 8, 'eval_duration': 251120000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e52-c221-7dd3-8b70-accd27aff336-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [2]:
def answer(country):
    '나라명을 입력받아 llm에게 수도명을 받아 return'
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm = ChatOllama(model='llama3.2:1b')
    prompt_template = PromptTemplate(template='What is the capital of {country}?',
                                    input_variables = ['country']
                                    )
    result = llm.invoke(prompt_template.invoke(country))
    return result.content

In [3]:
country = input("수도를 알고싶은 나라는(영어)? ")
answer(country)

수도를 알고싶은 나라는(영어)? korea


'The capital of Korea is Seoul.'

## 2) 메세지 기반 프롬프트 작성
- list of BaseMessage
- BaseMessages 상속받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage
- [BaseMessage객체, BaseMessage객체, BaseMessage객체, ...]

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, BaseMessage
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant !"), # llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Paris."),
    HumanMessage(content="What is the capital of Korea?"), # 질문 답변 예제(few shot)
]
llm.invoke(message_list)

AIMessage(content='Korea is actually a country, and it has its own capital. The capital of South Korea is Seoul, and the capital of North Korea is Pyongyang.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:18:25.0174525Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5611780700, 'load_duration': 3668829700, 'prompt_eval_count': 86, 'prompt_eval_duration': 772340000, 'eval_count': 32, 'eval_duration': 1165527000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e79-260c-7ac2-80d8-1efa857380cc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 32, 'total_tokens': 118})

## 3) ChatPromptTemplate 사용(추천 ; 확장성 용이)
- BaseMessage 리스트 -> 튜플리스트

In [8]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    SystemMessage(content="You are a helpful assistant !"),
    HumanMessage(content="What is the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of Italy is Paris."),
    HumanMessage(content="What is the capital of {country}?"),
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 : ', prompt)

프롬프트 :  messages=[SystemMessage(content='You are a helpful assistant !', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [9]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant !"),
    ("human", "hat is the capital of Italy?"),
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of Italy is Paris."),
    ("human", "What is the capital of {country}?")
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 : ', prompt)

프롬프트 :  messages=[SystemMessage(content='You are a helpful assistant !', additional_kwargs={}, response_metadata={}), HumanMessage(content='hat is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [13]:
# llm.invoke(chatPromptTemplate.invoke({'country':'Korea'}))
llm.invoke(chatPromptTemplate.invoke({'Korea'}))

AIMessage(content='I think there may be a mistake there. There is no country called "Korea". The correct country is South Korea. \n\nIf you\'re looking for the capital of South Korea, the correct answer is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:52:08.8613809Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1680523000, 'load_duration': 2109700, 'prompt_eval_count': 89, 'prompt_eval_duration': 142518000, 'eval_count': 44, 'eval_duration': 1531752000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e98-170a-7032-8c14-721039cf14e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 89, 'output_tokens': 44, 'total_tokens': 133})

# 3. 답변 형식 컨트롤하기
- invoke 샐행 결과 AIMessage() -> String, json 변환해주는 OutputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 이용하여 LLM출력(AIMessage)를 단순 문자열로 변환

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only',
    input_variables = ['country']
)
# 프롬프트 테플릿에 값 주입
prompt = prompt_template.invoke({'country':'Korea'})
print('프롬프트', prompt)
# llm에 질문
aimessage = llm.invoke(prompt)
# aimessage중 답변만 문자로 받기
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print('파서 결과 : ', result)

프롬프트 text='What is the capital of Korea? Return the name of the city only'
파서 결과 :  Seoul


In [19]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))
output_parser.invoke(llm.invoke(prompt_template.invoke({'Korea'})))

'Seoul'

In [23]:
chatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant !"),
    ("human", "hat is the capital of Italy?"),
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of Italy is Paris."),
    ("human", "What is the capital of {country}? Return the name of the city only.")
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})))

'Seoul'

## 2) Json 출력파서 이용
- {'name':'홍','age':20}

In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
llm = ChatOllama(model='llama3.2:1b')
country_datail_promrt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no addirional text.
    Example format :
    {{'Capital':'Seoul', 'Population':"50 million", 'Language':'Korean', 'Currency':'won'}}
    """,
    input_variavles = ['country']
)
prompt = country_datail_promrt.invoke({'country':'France'})
aimessage = llm.invoke(prompt)
output_parser = JsonOutputParser()
result = output_parser.invoke(aimessage)
print(type(result), result)

<class 'dict'> {'Capital': 'Paris', 'Population': '65 million', 'Language': 'French', 'Currency': 'Euro'}


In [10]:
info = output_parser.invoke(llm.invoke(country_datail_promrt.invoke('France')))
info

{'Capital': 'Paris',
 'Population': '65 million',
 'Language': 'French',
 'Currency': 'Euro'}

## 3) 구조화된 객체로 반환
- Pydantic 모델(pip show pydantic)을 사용하여 LLM출력을 구조화된 형식으로 받기(JsonParser 좀 안정적)
- Pydantic : 데이터 유효성 검가, 설정관리를 간편하게 해주는 라이브러리

In [13]:
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
user = User('1', "홍길동")
user = User(1, "홍")
print(user)
print(user.id, user.name, user.is_active)

1 홍 True


In [17]:
from pydantic import BaseModel, Field
class User(BaseModel):
    # gt=0 : id>0, lt=0 : id<0, ge=0 : id>=0, le=0 : id<=0
    id:int         = Field(gt=0,         description='id')
    name:str       = Field(min_length=2, description='name')
    is_active:bool = Field(default=True, description='ID활성화')
user = User(id="1", name='홍길동', is_active=True)
print(user)

id=1 name='홍길동' is_active=True


In [22]:
country_datail_promrt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no addirional text.
    Example format :
    {{'Capital':'Seoul', 'Population':"50 million", 'Language':'Korean', 'Currency':'won'}}
    """,
    input_variavles = ['country']
)
class CountryDetail(BaseModel):
    capital:str    = Field(description="the capital of the country")
    population:int = Field(description="the population of the country")
    language:str   = Field(description="the language of the country")
    currency:str   = Field(description="the currency of the country")
# 출력파서 + LLM 
structedllm = llm.with_structured_output(CountryDetail)
info = structedllm.invoke(country_datail_promrt.invoke({'country':'Korea'}))
print(type(info))
print(info)
print(info.capital, info.population, info.language, info.currency)
print(info.model_dump()) # 객체를 dict로

<class '__main__.CountryDetail'>
capital='Seoul' population=50 language='Korean' currency='won'
Seoul 50 Korean won
{'capital': 'Seoul', 'population': 50, 'language': 'Korean', 'currency': 'won'}


# 4. LCEL(LangChain Expression Language)응 활용한 렝체인 생성
## 1) 문자열 출력 파서 사용

## 2) LCEL을 사용한 체인구성